# Problem 2 on Homework 5

We are asked to show that if

$$F(a,b,n) = \prod_{i=1}^a \prod_{j=1}^b \frac{i+j+n-1}{i+j-1}$$

then

$$F(a,b,n) F(a,b,n-2) = F(a,b,n-1)^2 - F(a+1, b-1, n-1)F(a-1,b+1,n-1)$$

which is annoying, but still possible to do while still having confidence that you have done the algebra correctly.  I should have scaffolded the problem a little bit more.  Here is how I approach difficult computations such as this.

First of all, there are a few guiding principles.

1) There's probably not going to be a great, super-tidy way to do this - we need to find *a* way, while also being completely confident that it is correct.

2) Never do any algebra without simultaneously checking it on the computer, if you can possibly avoid it.

3) Success here (as with anything difficult) is not about how clever you are, it is about hitting the problem with the computer until it dies.

4) Our objective is to prove the theorem, while simultaneously building our confidence that it is true, as well as building our confidence in our own computations.
 

# Initial Computations

so.  First of all: I have given an exercise, to prove a certain result.  Is the result even true?  We know from some roundabout way that it is, but let's just check it directly.

I'm also going to keep track of my time.  I started at 8am on Friday. 

In [1]:
import itertools

def F(a,b,n):
    numerator = 1
    denominator = 1
    for i in range(1, a+1):
        for j in range(1, b+1):
            numerator *= i+j+n-1
            denominator *= i+j-1
    return numerator//denominator # the // does integer divide

for (a,b,n) in itertools.product(range(2,5), repeat=3):
    lhs = F(a,b,n) * F(a,b,n-2)
    rhs = F(a,b,n-1)**2 - F(a+1, b-1, n-1)*F(a-1, b+1, n-1)
    assert lhs==rhs, "Doesn't work when a,b,n = {}".format((a,b,n))

print("Looks good")

Looks good


Okay, now we believe the conjecture, without having to believe any other fancy facts from linear algebra.  Let's now set about simplifying it.  The goal here is to "get rid of" as many of the products as possible.  There's different approaches but let's just pick one: simplifying both sides of the desired identity repeatedly (and reversibly) until we get something we can actually prove.   

I'll start by dividing everything by $$F(a,b,n-1)^2$$ so what we need to prove is equivalent to $X_L = 1 - X_R$, where

$$X_L = \frac{F(a,b,n)F(a,b,n-2)}{F(a,b,n-1)^2}$$

and

$$X_R = \frac{F(a-1,b+1,n-1)F(a+1,b-1,n-1)}{F(a,b,n-1)^2}$$

and we're just going to try and simplify $X_L$ and $X_R$ as much as possible, to reduce the number of products.  As we do those tasks, we check everything we do on the computer for small examples.


# Simplifying $X_L$

My first thought is that $X_L = G(a,b,n)/G(a,b,n-1)$, where $G(a,b,n) = F(a,b,n)/F(a,b,n-1)$.  Let's simplify $G(a,b,n)$ while checking that it still is equal to the left side.  Note the denominators cancel exactly, and then one of the products (say, the inner one) is going to telescope:

\begin{align*}
G(a,b,n) &= \prod_{i=1}^a \prod_{j=1}^b \frac{(i+j+n-1)(i+j-1)}{(i+j-1)(i+j+n-2)} \\
&= \prod_{i=1}^a \prod_{j=1}^b\frac{i+j+n-1}{i+j+n-2} \\
&= \prod_{i=1}^a \frac{i+b+n-1}{i+n-1}.
\end{align*}

Did I do that correctly?  I think it is a theorem that I did not.  So, I write code to check that my simplified formula for $G$ still satisfies $X_L = G(a,b,n)/G(a,b,n-1)$ for small $a,b,n$, then simultaneously edit the code and the formula above until it is correct.  (The result: I'd written b instead of n at some point, so I fixed it, and now it says the above).

Note that everything in sight is a rational number so we'll use Sympy's Rational class to do arithmetic.

In [2]:
from sympy import Rational

def X_L(a,b,n):
    return Rational(F(a,b,n)*F(a,b,n-2), F(a,b,n-1)**2)

def G(a,b,n):
    numerator = 1
    denominator = 1
    for i in range(1, a+1):
        numerator *= i+b+n-1
        denominator *= i+n-1
    return Rational(numerator,denominator)

for a,b,n in itertools.product(range(2,5), repeat=3):
    assert G(a,b,n)/G(a,b,n-1) == X_L(a,b,n), "doesn't work when a,b,n={}".format((a,b,n ))

Having done that, it now looks like $G(a,b,n)/G(a,b,n-1)$ will telescope in a strikingly similar way:

\begin{align*}
\frac{G(a,b,n)}{G(a,b,n-1)} &= \prod_{i=1}^a \frac{(i+n-2)(i+b+n-1)}{(i+n-1)(i+b+n-2)} \\
&= \frac{(n-1)(a+b+n-1)}{(a+n-1)(b+n-1)}
\end{align*}

Is that right?  Again, I assume not, and check it for small $a,b,n$.  What I'd written was total crap.  So I did it again and tested it, and it was right. 

In [3]:
for a,b,n in itertools.product(range(2,5), repeat=3):
    assert G(a,b,n)/G(a,b,n -1) == Rational((n-1)*(a+b+n-1), (a+n-1)*(b+n-1)), "doesn't work when a,b,n={}".format((a,b,n ))


Amazing.  That is a rational function!  We have learned that if $X_L$ is as defined, then a simpler formula holds: $$X_L = \frac{(n-1)(a+b+n-1)}{(a+n-1)(b+n-1)}.$$
This gives us a lot of confidence that a theorem such as the one we're trying to prove might hold: our final task, once we simplify the right hand side, will be to show an identity of rational functions.  We know how to do that from college algebra!  

Note: it's 9:20am on Friday now, and I took 20 minutes to do a thing which wasn't math.  Time spent: 1 hour, going between writing markdown/LaTeX, writing code, and writing on my whiteboard in equal measure.

# Simplifying X_R

We start with our definition for $F$, which I'll copy paste here,

$$F(a,b,n) = \prod_{i=1}^a \prod_{j=1}^b \frac{i+j+n-1}{i+j-1}$$

and the definition for $X_R$, which is
$$X_R = \frac{F(a-1,b+1,n-1)F(a+1,b-1,n-1)}{F(a,b,n-1)^2}$$

Let's expand everything out and notice that, unlike the left side, we have to play around with the index sets in the products, since $a$ and $b$ are changing. 


Note: For this kind of stuff, it's WAY more reliable to copy-paste formulas from the previous line of TeX than it is to rewrite them out on the board.

\begin{align*}
X_R &= \frac{F(a-1,b+1,n-1)F(a+1,b-1,n-1)}{F(a,b,n-1)^2}\\
&= \left(\frac{\displaystyle \prod_{i=1}^{a-1} \prod_{j=1}^{b+1} \frac{i+j+n-2}{i+j-1}} 
{\displaystyle \prod_{i=1}^a \prod_{j=1}^b \frac{i+j+n-2}{i+j-1}} 
\right)
\cdot
\left(
\frac{\displaystyle \prod_{i=1}^{a+1} \prod_{j=1}^{b-1} \frac{i+j+n-2}{i+j-1}}
{\displaystyle \prod_{i=1}^a \prod_{j=1}^b \frac{i+j+n-2}{i+j-1}}\\
\right)
\end{align*}

There's a few moves I could do here - reindexing products and the like, but today I don't feel like doing any of that.  All the factors are identical as functions in (i,j,n), so I'm just going to look at the index sets of the products and cancel upstairs and downstairs.  For instance in that second factor:
- the terms for $i=a+1$ and $j=\{1, \dots, b-1\}$ appear only in the numerator,
- the terms for $j=b$ and $i=\{1,\dots, a\}$ appear only in the denominator,
- Every other term appears both upstairs and downstairs, so cancels on the nose.

So, I think that

$$
\frac{F(a+1,b-1,n-1)}{F(a,b,n-1)}
= 
\frac{\displaystyle  \prod_{j=1}^{b-1} \frac{a+j+n-1}{a+j}}
{\displaystyle \prod_{i=1}^a \frac{i+b+n-2}{i+b-1}} 
$$

Is that true?  Probably not, let's fix the problems.  This time I really screwed everything up, making a lot of twos into ones, so I did about 4 rounds of edits until the formula matched the code and the code didn't throw an exception.

In [4]:
for a,b,n in itertools.product(range(2,5), repeat=3):
    left_thing = Rational(F(a+1,b-1,n-1),F(a,b,n-1))
    right_thing = 1
    for j in range(1, b):
        right_thing *= Rational(a+j+n-1, a+j)
    for i in range(1, a+1):
        right_thing /= Rational(i+b+n-2, i+b-1)
    assert left_thing==right_thing, "doesn't work for (a,b,n)={}".format((a,b,n))
                

Similarly,  I think that in the first term,

- the terms for $j=b+1$ and $i=\{1, \dots, a-1\}$ appear only in the numerator,
- the terms for $i=a$ and $j=\{1,\dots, b\}$ appear only in the denominator,
- Every other term appears both upstairs and downstairs, so cancels on the nose.
and that means that I think that

$$
\frac{F(a-1,b+1,n-1)}{F(a,b,n-1)}
= 
\frac
{\displaystyle  \prod_{i=1}^{a-1} \frac{i+b+n-1}{i+b}}
{\displaystyle \prod_{j=1}^b \frac{a+j+n-2}{a+j-1}} 
$$
Did I do *that* right?  Amazing, I actually did that correctly.  Fantastic.  I will reward myself with a coffee.


In [5]:
for a,b,n in itertools.product(range(2,5), repeat=3):
    left_thing = Rational(F(a-1,b+1,n-1),F(a,b,n-1))
    right_thing = 1
    for i in range(1, a):
        right_thing *= Rational(i+b+n-1, i+b)
    for j in range(1, b+1):
        right_thing /= Rational(a+j+n-2, a+j-1)
    assert left_thing==right_thing, "doesn't work for (a,b,n)={}".format((a,b,n))

Coffee attained.  It's 10:00am and I spent another 10 minutes on not-math.

Continuing from my now-believed formula, putting it all together, I see that 

$$X_R =
\left( 
\frac{\displaystyle  \prod_{j=1}^{b-1} \frac{a+j+n-1}{a+j}}
{\displaystyle \prod_{i=1}^a \frac{i+b+n-2}{i+b-1}} 
\right)
\left(
\frac
{\displaystyle  \prod_{i=1}^{a-1} \frac{i+b+n-1}{i+b}}
{\displaystyle \prod_{j=1}^b \frac{a+j+n-2}{a+j-1}} 
\right)
$$

I can see right away I want to cancel the denominator of the first factor against the numerator of the second factor, as well as the other way around, so before thinking AT ALL I am going to reindex both denominator products so that the terms are the same:
$$X_R =
\left( 
\frac{\displaystyle  \prod_{j=1}^{b-1} \frac{a+j+n-1}{a+j}}
{\displaystyle \prod_{i=0}^{a-1} \frac{i+b+n-1}{i+b}} 
\right)
\left(
\frac
{\displaystyle  \prod_{i=1}^{a-1} \frac{i+b+n-1}{i+b}}
{\displaystyle \prod_{j=0}^{b-1} \frac{a+j+n-1}{a+j}} 
\right)
$$
and we immediately see that the only terms which survive are $i=0, j=0$ in the denominator.  It seems that I believe that
$$X_R =
\frac{a}{a+n-1} \cdot
\frac{b}{b+n-1} 
$$
Checking that: I initially forgot to invert the fractions, but that was my only error.  It's 10:20.




In [6]:
for a,b,n in itertools.product(range(2,5), repeat=3):
    X_R = Rational(F(a-1,b+1,n-1)*F(a+1,b-1,n-1), F(a,b,n-1)**2)
    assert X_R == Rational(a*b, (a+n-1) * (b+n-1)), "doesn't work when a,b,n={}".format((a,b,n))

# Finishing it off

The desired theorem is equivalent to the identity $X_L = 1-X_R$ as stated at the top.  Moreover, our carefully checked simplifications show that 

$$X_L = \frac{(n-1)(a+b+n-1)}{(a+n-1)(b+n-1)}, \qquad X_R =
\frac{ab}{(a+n-1)(b+n-1)} 
$$

This is an identity of rational functions, hence should be trivial.  We compute $X_L + X_R$ as a rational function of $a,b,n$.  If we get 1, then the theorem is proven.

In [7]:
from sympy import symbols
(a,b,n) = symbols("a,b,n")
X_L = (n-1)*(a+b+n-1)/(a+n-1)/(b+n-1)
X_R = a*b/(a+n-1) / (b+n-1)

(X_L+X_R).simplify()

1

QED.

# Conclusion

The time 10:30am.  So, 2 hours of math, 30 minutes of interruptions.  Note: I didn't do any work on paper, and I didn't rough draft any of the algebra, I didn't consult any references.  

So, in the end, I stand by what I said: it was annoying, but not out of line, to do this problem.

Also note: in the above solution, I didn't do anything particularly clever, and I did plenty of things that are monumentally stupid!  However: **I caught all my mistakes before I proceeded any further**.  That both saved me time, and produced a correct result.  This is the standard needed for doing a difficult proof, in any area of mathematics.  